# 3. Weryfikacja poprawności typu IfcEntity

Używamy wytrenowanego Random Forest (z notebooka 02) do przewidzenia
"prawdziwej" klasy każdego elementu na podstawie samej geometrii, a następnie
porównujemy to z typem encji faktycznie zapisanym w pliku IFC.

Wynik: `data/entity_qa_report.csv` — lista elementów, dla których zapisany
typ IFC wygląda na błędny, wraz z sugerowaną (na podstawie geometrii)
poprawną klasą.

In [1]:
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd()
DATA_CSV = ROOT / "data" / "ifc_features.csv"
MODELS_DIR = ROOT / "models"
OUT_CSV = ROOT / "data" / "entity_qa_report.csv"

NON_FEATURE_COLS = ["file", "split", "label", "stored_ifc_class", "type_mismatch", "guid", "name"]
LABEL_TO_IFC_CLASS = {"Beam": "IfcBeam", "Slab": "IfcSlab", "Stair": "IfcStair", "Wall": "IfcWall"}

rf = joblib.load(MODELS_DIR / "random_forest.joblib")
feature_cols = joblib.load(MODELS_DIR / "feature_cols.joblib")

df = pd.read_csv(DATA_CSV)
X = df[feature_cols].values

In [2]:
proba = rf.predict_proba(X)
pred = rf.classes_[proba.argmax(axis=1)]
confidence = proba.max(axis=1)

df["predicted_label"] = pred
df["predicted_confidence"] = confidence
df["predicted_ifc_class"] = df["predicted_label"].map(LABEL_TO_IFC_CLASS)

# Does the stored entity type in the file match what the geometry says it should be?
df["stored_type_looks_wrong"] = df["stored_ifc_class"] != df["predicted_ifc_class"]
# Does the geometry model disagree with the curated ground-truth folder label?
# (in-sample for train rows, held-out generalisation for test rows)
df["disagrees_with_ground_truth"] = df["predicted_label"] != df["label"]

report_cols = [
    "file", "split", "label", "stored_ifc_class", "type_mismatch",
    "predicted_label", "predicted_ifc_class", "predicted_confidence",
    "stored_type_looks_wrong", "disagrees_with_ground_truth",
]
df[report_cols].to_csv(OUT_CSV, index=False)
print(f"wrote {len(df)} rows -> {OUT_CSV}")

wrote 1378 rows -> C:\Users\agata\OneDrive\Desktop\Test\ifc_test\data\entity_qa_report.csv


In [3]:
n = len(df)
n_proxy = (df["stored_ifc_class"] == "IfcBuildingElementProxy").sum()
n_proxy_fixed = (
    (df["stored_ifc_class"] == "IfcBuildingElementProxy")
    & (df["predicted_label"] == df["label"])
).sum()
n_disagree = df["disagrees_with_ground_truth"].sum()

print(f"elementy zapisane jako generyczny IfcBuildingElementProxy: {n_proxy}")
print(f"  -> model geometryczny odzyskuje poprawna klase dla {n_proxy_fixed}/{n_proxy} z nich")
print(f"elementy, dla ktorych model nie zgadza sie z etykieta (folderem): {n_disagree}/{n}")

elementy zapisane jako generyczny IfcBuildingElementProxy: 209
  -> model geometryczny odzyskuje poprawna klase dla 206/209 z nich
elementy, dla ktorych model nie zgadza sie z etykieta (folderem): 19/1378


## Przykladowe oflagowane elementy (zapisany typ wyglada podejrzanie)

In [4]:
flagged = df[df["stored_type_looks_wrong"]].sort_values("predicted_confidence", ascending=False)
flagged[["file", "label", "stored_ifc_class", "predicted_ifc_class", "predicted_confidence"]].head(15)

,file,label,stored_ifc_class,predicted_ifc_class,predicted_confidence
138,bfe661448de24e999b6233402f6044f6.ifc,Beam,IfcBuildingElementProxy,IfcBeam,1.0
126,b421c3b9d38d4359a95786d3e45a580d.ifc,Beam,IfcBuildingElementProxy,IfcBeam,1.0
133,bb6dedc0088d435ab7c80be8ab55de33.ifc,Beam,IfcBuildingElementProxy,IfcBeam,1.0
109,94455c146d0a4de184a0fd883086093d.ifc,Beam,IfcBuildingElementProxy,IfcBeam,1.0
301,0e1cca4df109419c82833f8d26d5515c.ifc,Slab,IfcBuildingElementProxy,IfcSlab,1.0
303,102f7abd60774c57afc2cdb3932f3537.ifc,Slab,IfcBuildingElementProxy,IfcSlab,1.0
185,f52250cc65f249fc9f50ca2ea221f97c.ifc,Beam,IfcBeamStandardCase,IfcBeam,1.0
178,eaccde337b534eafb766345f81c166e2.ifc,Beam,IfcBeamStandardCase,IfcBeam,1.0
149,cc87cf97e6fe422db29b2cee72d3de8b.ifc,Beam,IfcBuildingElementProxy,IfcBeam,1.0
290,07e54ebc3b8848f488973b9e2c77e075.ifc,Slab,IfcBuildingElementProxy,IfcSlab,1.0
